In [14]:
import pandas as pd
import numpy as np
import re
import sqlite3

Fills SQLite database 'countries.db' with data from Findex DB.
Database 'countries.db' is assumed to be empty of data at this point.

Data is stored in series named according to databank.worldbank.org even though the database is populated with data from World Bank Open Data (https://data360.worldbank.org/en/dataset/WB_FINDEX).
Some series are no longer available (Worried part) and file downloaded from databank.worldbank.org is used.

In [15]:
db_name = "../data/processed/countries.db"

In [16]:
def insert_data(db_name: str, 
                df: pd.DataFrame, 
                table_name: str):
    with sqlite3.connect(db_name) as countries_conn:
        countries_conn.execute("pragma foreign_keys = on;")
        df.to_sql(table_name, 
              countries_conn,
              if_exists = 'append',
              index = False)

# Worried (from databank.worldbank.org)
From a database which is no longer available

Data from database: Global Financial Inclusion
Last Updated: 03/28/2024

In [18]:
'''
source_code = 'WB'
source_name = 'World Bank'
conn = sqlite3.connect(db_name)
conn.execute('pragma foreign_keys = on;')
conn.execute(
    """
    INSERT INTO sources (
        source_code,
        source_name
    )
    VALUES (?, ?);
    """,
    (
        source_code,
        source_name
    )
)
conn.commit()
conn.close()
'''

'\nsource_code = \'WB\'\nsource_name = \'World Bank\'\nconn = sqlite3.connect(db_name)\nconn.execute(\'pragma foreign_keys = on;\')\nconn.execute(\n    """\n    INSERT INTO sources (\n        source_code,\n        source_name\n    )\n    VALUES (?, ?);\n    """,\n    (\n        source_code,\n        source_name\n    )\n)\nconn.commit()\nconn.close()\n'

In [19]:
data_worried = pd.read_csv("../data/raw/48ebe16e-d822-45d5-8f79-2502c8a51b05_Series - Metadata.csv", sep = ',', nrows=28705)

## Worried series (codes and names)

In [20]:
data_series = data_worried[["Series Code", "Series Name"]].dropna().drop_duplicates()

In [21]:
series = data_series[["Series Code", "Series Name"]].drop_duplicates()
series.rename(columns = {"Series Code": "series_id", "Series Name": "series_description"}, inplace = "True")
series["source_code"] = "WB"
#insert_data(db_name, series, "series")

## Worried series countries

In [22]:
worried_countries = data_worried[["Country Code", "Country Name"]].dropna().drop_duplicates()

In [23]:
regions = [
        ['ARB', 'Arab World'],
       ['EAS', 'East Asia & Pacific'],
       ['EAP', 'East Asia & Pacific (excluding high income)'],
       ['EMU', 'Euro area'],
       ['ECS', 'Europe & Central Asia'],
       ['ECA', 'Europe & Central Asia (excluding high income)'],
       ['HIC', 'High income'],
       ['LCN', 'Latin America & Caribbean'],
       ['LAC', 'Latin America & Caribbean (excluding high income)'],
       ['LMY', 'Low & middle income'],
       ['LIC', 'Low income'],
       ['LMC', 'Lower middle income'],
       ['MEA', 'Middle East & North Africa'],
       ['MNA', 'Middle East & North Africa (excluding high income)'],
       ['MIC', 'Middle income'],
       ['NAC', 'North America'],
       ['OED', 'OECD members'],
       ['SAS', 'South Asia'],
       ['SSF', 'Sub-Saharan Africa'],
       ['SSA', 'Sub-Saharan Africa (excluding high income)'],
       ['UMC', 'Upper middle income'],
       ['WLD', 'World']]
regions_df = pd.DataFrame(regions, columns = ['iso3_id', 'name'])

In [24]:
codes_to_drop = regions_df['iso3_id']
countries_to_insert = worried_countries[~worried_countries['Country Code'].isin(codes_to_drop)].copy()
countries_to_insert.rename(columns = {'Country Code': 'iso3_id', 'Country Name': 'country_name'}, inplace = 'True')

In [25]:
#insert_data(db_name, countries_to_insert, "countries")

## Waves

In [26]:
waves_to_insert = pd.DataFrame(columns = ['wave_name', 'wave_description', 'start_date', 'end_date', 'waves_date_precision', 'source_code'])

In [27]:
waves_to_insert['wave_name'] = [2011, 2014, 2017, 2021, 2024]

waves_to_insert['start_date'] = waves_to_insert['wave_name']
waves_to_insert['start_date'] = waves_to_insert['start_date'].apply(lambda y: str(y) + '-01-01')

waves_to_insert['end_date'] = waves_to_insert['wave_name']
waves_to_insert.loc[3, 'end_date'] = 2022
waves_to_insert['end_date'] = waves_to_insert['end_date'].apply(lambda y: str(y) + '-01-01')

# descriptions are taken from the Methodology sections of wave reports
waves_to_insert.loc[0, 'wave_description'] = ( 
    'The indicators in the Global Financial Inclusion (Global Findex) Database are drawn from survey data covering more than 150,000 people in 148 economies representing'
    'more than 97 percent of the world’s population. The target population is the entire civilian, noninstitutionalized population age 15 and older.'
    'Surveys are conducted in the major languages of each economy.')
waves_to_insert.loc[1, 'wave_description'] = (
    'The indicators in the 2014 Global Financial Inclusion (Global Findex) database are drawn from survey data covering almost 150,000 people in 143 economies—representing'
    'more than 97 percent of the world’s population. The target population is the entire civilian, noninstitutionalized population age 15 and above.')
waves_to_insert.loc[2, 'wave_description'] = (
    'The indicators in the 2017 Global Findex database are drawn from survey data covering almost 150,000 people in 144 economies—representing more than 97 percent'
    'of the world’s population. The target population is the entire civilian, noninstitutionalized population age 15 and above.')
waves_to_insert.loc[3, 'wave_description'] = (
    'The indicators in the Global Findex 2021 database are drawn from survey data covering almost 128,000 people in 123 economies, representing 91 percent of the world’s population.'
    'The target population is the entire civilian, noninstitutionalized population age 15 and up.')
waves_to_insert.loc[4, 'wave_description'] = (
    'The indicators in the Global Findex Database 2025 are drawn from survey data covering more than 140,000 people in 141 economies, representing 96 percent'
    'of the world’s population. The target population is the entire noninstitutionalized civilian population ages 15 and up.')

waves_to_insert["waves_date_precision"] = 'year'
waves_to_insert['source_code'] = 'WB'
waves_to_insert

,wave_name,wave_description,start_date,end_date,waves_date_precision,source_code
0,2011,The indicators in the Global Financial Inclusi...,2011-01-01,2011-01-01,year,WB
1,2014,The indicators in the 2014 Global Financial In...,2014-01-01,2014-01-01,year,WB
2,2017,The indicators in the 2017 Global Findex datab...,2017-01-01,2017-01-01,year,WB
3,2021,The indicators in the Global Findex 2021 datab...,2021-01-01,2022-01-01,year,WB
4,2024,The indicators in the Global Findex Database 2...,2024-01-01,2024-01-01,year,WB


In [28]:
#insert_data(db_name, waves_to_insert, "waves")

## Worried values

In [29]:
data_worried.rename(columns = {'2011 [YR2011]': 2011, '2014 [YR2014]': 2014, '2017 [YR2017]': 2017, '2021 [YR2021]': 2021, '2022 [YR2022]': 2022}, inplace = True)
data_worried.dropna(inplace = True)
worried_values = data_worried[~data_worried['Country Code'].isin(codes_to_drop)].copy()
print((worried_values[2022] != '..').sum())
print((worried_values[2021] != '..').sum())
print((worried_values[2017] != '..').sum())
print((worried_values[2014] != '..').sum())
print((worried_values[2011] != '..').sum())

2376
17592
0
0
0


In [31]:
values_to_insert_2022 = pd.DataFrame(columns = ["series_value", "observation_date", "sv_date_precision", "series_id", "source_code", "iso3_id", "wave_id"])
values_to_insert_2021 = pd.DataFrame(columns = ["series_value", "observation_date", "sv_date_precision", "series_id", "source_code", "iso3_id", "wave_id"])

In [32]:
values_to_insert_2022['series_value'] = pd.to_numeric(worried_values[2022], errors = 'coerce')
values_to_insert_2022['observation_date'] = '2022-01-01'
values_to_insert_2022['sv_date_precision'] = 'year'
values_to_insert_2022['series_id'] = worried_values['Series Code']
values_to_insert_2022['source_code'] = 'WB'
values_to_insert_2022['iso3_id'] = worried_values['Country Code']

values_to_insert_2021['series_value'] = pd.to_numeric(worried_values[2021], errors = 'coerce')
values_to_insert_2021['observation_date'] = '2021-01-01'
values_to_insert_2021['sv_date_precision'] = 'year'
values_to_insert_2021['series_id'] = worried_values['Series Code']
values_to_insert_2021['source_code'] = 'WB'
values_to_insert_2021['iso3_id'] = worried_values['Country Code']

In [33]:
values_to_insert_2022.dropna(subset=['series_value'], inplace = True)
print(len(values_to_insert_2022))
values_to_insert_2021.dropna(subset=['series_value'], inplace = True)
print(len(values_to_insert_2021))
print(len(values_to_insert_2022) + len(values_to_insert_2021))

2376
17592
19968


In [35]:
#insert_data(db_name, values_to_insert_2022, "series_values")
#insert_data(db_name, values_to_insert_2021, "series_values")

In [36]:
'''
with sqlite3.connect(db_name) as conn:
    conn.execute('pragma foreign_keys = on;')
    cursor = conn.execute("select wave_id from waves where source_code = 'WB' and wave_name = '2021'")
    wave_id = cursor.fetchone()[0]
    conn.execute(
        """
        update series_values
        set wave_id = ?
        where source_code = 'WB'
            and observation_date >= '2021-01-01'
            and observation_date <= '2022-01-01';
        """,
        (wave_id,)
    )
'''

## Stratification codes from databank.worldbank.org Findex

In [37]:
main_code = []
strat_code = []
main_name = []
strat_name = []
age_strat_name = []
for row_no in data_series.index:
    code = data_series.iloc[row_no]["Series Code"]
    m = re.match(r'^([A-Za-z0-9.]+?)(?:\.([0-9]+))?$', code)
    main_code.append(m.group(1))
    strat_code.append(m.group(2))
    
    name = data_series.iloc[row_no]["Series Name"]
    m = re.match(r'([A-Za-z ]+:[A-Za-z ]+)(?:,\s*)?(.*)\((.*)\)', name)
    main_name.append(m.group(1))
    strat_name.append(m.group(2))
    age_strat_name.append(m.group(3))

data_series["Main Code"] = main_code
data_series["Strat Code"] = strat_code
data_series["Main Name"] = main_name
data_series["Strat Name"] = strat_name
data_series["Age Strat Name"] = age_strat_name

In [38]:
stratification_codes = data_series[["Strat Code", "Strat Name", "Age Strat Name"]].copy()
stratification_codes.drop_duplicates(inplace = True)
stratification_codes.sort_values(by = ["Strat Code"], inplace = True)
stratification_codes

,Strat Code,Strat Name,Age Strat Name
1,1,female,% age 15+
11,10,urban,% age 15+
7,11,out of labor force,% age 15+
2,12,in labor force,% age 15+
5,2,male,% age 15+
12,3,young,% ages 15-24
6,4,older,% age 25+
8,5,primary education or less,% ages 15+
10,6,secondary education or more,% ages 15+
3,7,"income, poorest 40%",% ages 15+


# Findex DB from data360 World Bank

In [39]:
findex = pd.read_csv("../data/raw/WB_FINDEX_20_IV_2026.csv")

In [40]:
findex.drop(columns = ['STRUCTURE',    # datastructure
                       'STRUCTURE_ID', # WB.DATA360:DS_DATA360(1.3)
                       'ACTION',       # I
                       'FREQ',         # A3 
                       'UNIT_TYPE',    # RATIO
                       'DATABASE_ID',  # WB_FINDEX
                       'TIME_FORMAT',  # 602
                       'UNIT_MULT',    # 0
                       'DATA_SOURCE',  # WB_FINDEX
                       'OBS_CONF',     # PU
                       'OBS_STATUS',   # A
                       'FREQ_LABEL',   # Triennial
                       'OBS_CONF_LABEL',# Public
                       'DATA_SOURCE_LABEL', # Global Findex Database
                       'OBS_STATUS_LABEL', # Normal value
                       'UNIT_MULT_LABEL', # Units
                       'TIME_FORMAT_LABEL', # CCYY
                       'DATABASE_ID_LABEL', # Global Findex Database
                       'UNIT_TYPE_LABEL' # Ratio
                      ], inplace = True)

In [41]:
findex = findex.loc[~findex['REF_AREA'].isin(codes_to_drop)]

REF_AREA in original db has 173 codes and this includes aggregates

INDICATOR has 280 codes

UNIT_MEASURE: additional info stored in databank.worldbank.org as .s "subseries",
usually percentage of responders who answered who have some feature (like own a phone or set a password),
it's combined with no additional stratification

TIME_PERIOD seems to be wave/year of observation

OBS_VALUE is REAL in the DB (all numbers)

comp_breakdown_1
_T	Total
INCOME_GE_Q3	Income Level: Richest 60%
INCOME_Q1T2	Income Level: Poorest 40%
DL_P_NDSD	Level of Difficulty: Possible and not difficult or somewhat difficult
DL_P_VD	Level of Difficulty: Possible and very difficult

comp_breakdown_2
_T	Total
LFP_OUTWORK	Employment: Out of the workforce/labor force
LFP_INWORK	Employment: In the workforce/labor force

comp_breakdown_3
_T	Total
ISCED11_GE_2_3	Education: Secondary education or more
ISCED11_0T1	Education: Primary education or less

Stratification into 13 groups (total and 12 stratified) + one additional info series (13-14 values per country per time period) (TODO CHECK!!!)

In [42]:
findex.info()

<class 'pandas.core.frame.DataFrame'>
Index: 349296 entries, 0 to 381789
Data columns (total 20 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   REF_AREA                349296 non-null  object 
 1   INDICATOR               349296 non-null  object 
 2   SEX                     349296 non-null  object 
 3   AGE                     349296 non-null  object 
 4   URBANISATION            349296 non-null  object 
 5   UNIT_MEASURE            349296 non-null  object 
 6   COMP_BREAKDOWN_1        349296 non-null  object 
 7   COMP_BREAKDOWN_2        349296 non-null  object 
 8   COMP_BREAKDOWN_3        349296 non-null  object 
 9   TIME_PERIOD             349296 non-null  int64  
 10  OBS_VALUE               349296 non-null  float64
 11  REF_AREA_LABEL          349296 non-null  object 
 12  INDICATOR_LABEL         349296 non-null  object 
 13  SEX_LABEL               349296 non-null  object 
 14  AGE_LABEL               3

# Countries

In [43]:
findex_countries = findex[["REF_AREA", "REF_AREA_LABEL"]].drop_duplicates()

In [44]:
#findex_countries.drop(findex_countries["REF_AREA_LABEL"][findex_countries["REF_AREA_LABEL"].str.contains("income")].index, inplace = True)
#findex_countries.drop(findex_countries["REF_AREA_LABEL"][findex_countries["REF_AREA_LABEL"].str.contains("World")].index, inplace = True)
#findex_countries.drop(findex_countries["REF_AREA_LABEL"][findex_countries["REF_AREA_LABEL"].str.contains("Asia")].index, inplace = True)

In [45]:
findex_countries["REF_AREA"]

0        ZMB
2        ZWE
4        AFG
6        ALB
8        DZA
        ... 
4580     ISL
8466     OMN
60136    QAT
78751    DJI
78836    SYR
Name: REF_AREA, Length: 162, dtype: object

# Series

## Main series idicators, no stratification
Between Data 360 and databank.worldbank.org for series indicators underscores become dots apart from series in suspicious_names.

In [46]:
#series_indicator_findex = pd.DataFrame(findex["INDICATOR"].unique(), columns = ['Findex Name'])
series_indicator_findex = findex[['INDICATOR', 'INDICATOR_LABEL']].drop_duplicates().copy()
series_indicator_findex.rename(columns = {'INDICATOR': 'Findex Indicator', 'INDICATOR_LABEL': 'Indicator Label'}, inplace = True)

In [48]:
series_indicator_findex['Findex Indicator Sub'] = series_indicator_findex['Findex Indicator'].apply(lambda s: re.match("(WB_FINDEX_)(.*)", s).group(2)).unique()

In [49]:
# all indicators which end with substring "_number" and "_number_letter"
# they need the underscore preserved to be compatible with databank.worldbank.org
suspicious_names = series_indicator_findex['Findex Indicator Sub'].str.extract("(.*_[0-9])$").dropna().sort_values(by = [0])
suspicious_names = pd.concat((suspicious_names, series_indicator_findex['Findex Indicator Sub'].str.extract("(.*_[0-9][A-Z])$").dropna()))
suspicious_names = pd.concat((suspicious_names, series_indicator_findex['Findex Indicator Sub'].str.extract("(.*_VD)$").dropna()))
suspicious_names = pd.concat((suspicious_names, series_indicator_findex['Findex Indicator Sub'].str.extract("(.*SD_ND)$").dropna()))
suspicious_names.reset_index(drop = True, inplace = True)

In [50]:
series_indicator_findex['Databank Indicator'] = series_indicator_findex['Findex Indicator Sub'].copy()

mask = ~series_indicator_findex['Databank Indicator'].isin(suspicious_names[0])
series_indicator_findex['Databank Indicator'].loc[mask] = series_indicator_findex['Databank Indicator'].loc[mask].str.replace('_', '.', regex = False)
series_indicator_findex['Databank Indicator'] = series_indicator_findex['Databank Indicator'].str.lower()
series_indicator_findex.sort_values(by = ['Findex Indicator Sub'], inplace = True)
series_indicator_findex.reset_index(drop = True, inplace = True)

## Indicators with stratification

In [51]:
stratification_codes['Strat Name'] = stratification_codes['Strat Name'].str.strip()
stratification_codes['Strat Code'] = '.' + stratification_codes['Strat Code']
stratification_codes.reset_index(drop = True, inplace = True)

In [52]:
# No stratification
stratification_codes.loc[12, "Strat Name"] = 'total'
stratification_codes.loc[12, "Strat Code"] = ''

# Some series have additional info provided about respondents
stratification_codes.loc[len(stratification_codes)] = {'Strat Code': '.s', 'Strat Name': 'additional info', 'Age Strat Name': '% age 15+'}

# One series with indicator FIN24OTHER (Main source of emergency funds in 30 days: other) is organized differently
# It has two stratifications (as below)
stratification_codes.loc[len(stratification_codes)] = {'Strat Code': '_SD_ND', 'Strat Name': 'possible and not difficult or somewhat difficult', 'Age Strat Name': '% age 15+'}
stratification_codes.loc[len(stratification_codes)] = {'Strat Code': '_VD', 'Strat Name': 'possible and very difficult', 'Age Strat Name': '% age 15+'}

stratification_codes['Age Strat Name'] =  '(' + stratification_codes['Age Strat Name'] + ')'

In [53]:
findex_female_mask = findex['SEX'] == 'F'
findex_male_mask = findex['SEX'] == 'M'
findex_young_mask = findex['AGE'] == 'Y15T24'
findex_older_mask = findex['AGE'] == 'Y_GE25'
#findex_ge15_mask = findex['AGE'] == 'Y_GE15'
findex_primary_education_mask = findex['COMP_BREAKDOWN_3'] == 'ISCED11_0T1'
findex_secondary_education_mask = findex['COMP_BREAKDOWN_3'] == 'ISCED11_GE_2_3'
findex_poorest_mask = findex['COMP_BREAKDOWN_1'] == 'INCOME_Q1T2'
findex_richest_mask = findex['COMP_BREAKDOWN_1'] == 'INCOME_GE_Q3'
findex_rural_mask = findex['URBANISATION'] == 'RUR'
findex_urban_mask = findex['URBANISATION'] == 'URB'
findex_out_of_work_mask = findex['COMP_BREAKDOWN_2'] == 'LFP_OUTWORK'
findex_employed_mask = findex['COMP_BREAKDOWN_2'] == 'LFP_INWORK'
findex_s_mask = ~(findex['UNIT_MEASURE'] == 'PT_RESP')
findex_total_mask = ((findex['SEX'] == '_T') 
     & (findex['AGE'] == 'Y_GE15') 
     & (findex['COMP_BREAKDOWN_1'] == '_T')
     & (findex['COMP_BREAKDOWN_2'] == '_T')
     & (findex['COMP_BREAKDOWN_3'] == '_T')
     & (findex['URBANISATION'] == '_T')
     & (findex['UNIT_MEASURE'] == 'PT_RESP'))
findex_difficult_mask = findex['COMP_BREAKDOWN_1'] == 'DL_P_VD'
findex_easy_mask = findex['COMP_BREAKDOWN_1'] == 'DL_P_NDSD'

In [54]:
masks = {
    # findex['SEX'] == 'F'
    # it is uniquely identifying for rows with female stratification
    'female': findex_female_mask,
    # findex['SEX'] == 'M'
    # it is uniquely identifying for rows with male stratification
    'male': findex_male_mask,
    # findex['AGE'] == 'Y15T24'
    # is uniquely identifying for rows with young stratification
    'young': findex_young_mask,
    # findex['AGE'] == 'Y_GE25'
    # is uniquely identifying for rows with older stratification
    'older': findex_older_mask,
    # findex['COMP_BREAKDOWN_3'] == 'ISCED11_0T1'
    # primary education or less, uniquely identified
    'primary education or less': findex_primary_education_mask,
    # findex['COMP_BREAKDOWN_3'] == 'ISCED11_GE_2_3'
    # secondary education or more, uniquely identified
    'secondary education or more': findex_secondary_education_mask,
    # findex['COMP_BREAKDOWN_1'] == 'INCOME_Q1T2'
    # income, poorest 40% , uniquely identified
    'income, poorest 40%': findex_poorest_mask,
    # findex['COMP_BREAKDOWN_1'] == 'INCOME_GE_Q3'
    # income, richest 60%, , uniquely identified
    'income, richest 60%': findex_richest_mask,
    # findex['URBANISATION'] == 'RUR'
    # rural, uniquely identified
    'rural': findex_rural_mask,
    # findex['URBANISATION'] == 'URB'
    # urban, uniquely identified
    'urban': findex_urban_mask,
    # findex['COMP_BREAKDOWN_2'] == 'LFP_OUTWORK'
    # out of labor force
    'out of labor force': findex_out_of_work_mask,
    # findex['COMP_BREAKDOWN_2'] == 'LFP_INWORK'
    # in labor force
    'in labor force': findex_employed_mask,
    # ~(findex['UNIT_MEASURE'] == 'PT_RESP')
    # this selects additional info provided for some series
    # not an actual stratification
    'additional info': findex_s_mask,
    # unstratified data
    'total': findex_total_mask,
    # stratification for FIN24OTHER
    'possible and not difficult or somewhat difficult': findex_easy_mask,
    'possible and very difficult': findex_difficult_mask
}

In [55]:
with sqlite3.connect(db_name) as conn:
    conn.execute('pragma foreign_keys = on;')
    cursor = conn.execute('select * from waves where source_code = "WB"')
    findex_waves = cursor.fetchall()
findex_waves = {pd.to_numeric(e[1]): e[0] for e in findex_waves}

In [57]:
fin_index = np.hstack((np.arange(len(findex)).reshape(-1,1), np.zeros(len(findex), dtype = int).reshape(-1,1)))
for key in masks.keys():
    print('---' + key + '---')

    fin_index[masks[key]] += 1

    # value, date, indicator, country
    # Intentional NOT .copy(). It's better if it's a view because less memory is used
    # pd.merge below creates a new DataFrame and only this one is later modified
    findex_values = findex[masks[key]][['OBS_VALUE', 'TIME_PERIOD', 'INDICATOR', 'REF_AREA']]
    # join to get names in databank form and names
    findex_values = pd.merge(findex_values, series_indicator_findex, how = 'left', left_on = "INDICATOR", right_on = "Findex Indicator")
    findex_values.drop(columns = ['INDICATOR', 'Findex Indicator', 'Findex Indicator Sub'], inplace = True)
    # assign a stratification code appropriate for the mask
    findex_values = findex_values.assign(**stratification_codes[stratification_codes['Strat Name'] == key].to_dict(orient = 'records')[0])
    # stratified indicator
    findex_values['series_id'] = findex_values['Databank Indicator'] + findex_values['Strat Code']
    findex_values.drop(columns = ['Databank Indicator', 'Strat Code'], inplace = True)

    series_to_insert = findex_values[['series_id', 'Indicator Label', 'Strat Name', 'Age Strat Name']].copy()
    series_to_insert.drop_duplicates(inplace = True)
    # stratified indicator labels
    labels = series_to_insert['Indicator Label'] + ', ' + series_to_insert['Strat Name'] + ' ' + series_to_insert['Age Strat Name']
    series_to_insert = pd.DataFrame({'series_id': series_to_insert['series_id'], 'series_description': labels})
    series_to_insert['source_code'] = 'WB'
    insert_data(db_name, series_to_insert, "series")
    print("No of series for " + key  + " stratification: " + str(len(series_to_insert)))

    findex_values.drop(columns = ['Indicator Label', 'Strat Name', 'Age Strat Name'], inplace = True)
    findex_values.rename(columns = {'OBS_VALUE': 'series_value', 'TIME_PERIOD': 'observation_date', 'REF_AREA': 'iso3_id'}, inplace = True)
    findex_values['wave_id'] = findex_values['observation_date']
    findex_values.loc[findex_values['wave_id'] == 2022, 'wave_id'] = 2021
    findex_values['wave_id'] = findex_values['wave_id'].apply(lambda wid: findex_waves[wid])
    findex_values['observation_date'] = findex_values['observation_date'].apply(lambda y: str(y) + '-01-01')
    findex_values['sv_date_precision'] = 'year'
    findex_values['source_code'] = 'WB'
    insert_data(db_name, findex_values, "series_values")

print("\nSummary:")
print(str(fin_index[:, 1].sum()) + " out of " + str(len(findex)) + " values seen.")
print(str(np.round(fin_index[:, 1].sum()/len(findex)*100, 2)) + "% values seen.")

if (fin_index[:, 1] > 1).sum():
    print("\nError: Stratification is not unique.\n")

not_read_mask = fin_index[:, 1] == 0
if not_read_mask.sum() > 0:
    print('\nRows which were not read:\n')
    print(findex[fin_index[:, 1] == 0])

---female---
No of series for female stratification: 244
---male---
No of series for male stratification: 244
---young---
No of series for young stratification: 244
---older---
No of series for older stratification: 244
---primary education or less---
No of series for primary education or less stratification: 244
---secondary education or more---
No of series for secondary education or more stratification: 244
---income, poorest 40%---
No of series for income, poorest 40% stratification: 242
---income, richest 60%---
No of series for income, richest 60% stratification: 242
---rural---
No of series for rural stratification: 237
---urban---
No of series for urban stratification: 237
---out of labor force---
No of series for out of labor force stratification: 244
---in labor force---
No of series for in labor force stratification: 244
---additional info---
No of series for additional info stratification: 147
---total---
No of series for total stratification: 280
---possible and not diffic